In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json

df = pd.read_csv("../data/processed/train_scaled.csv")

with open("../data/processed/top_sensors.json") as f:
    sensors = json.load(f)

In [2]:
from sklearn.model_selection import train_test_split

engine_ids = df["engine_id"].unique()

train_engines, test_engines = train_test_split(
    engine_ids, test_size=0.2, random_state=42
)

train_df = df[df["engine_id"].isin(train_engines)]
test_df = df[df["engine_id"].isin(test_engines)]

In [3]:
def create_sequences(df, sensors, window):

    X = []
    y = []

    for engine in df["engine_id"].unique():

        engine_df = df[df["engine_id"] == engine].sort_values("cycle")

        sensor_vals = engine_df[sensors].values
        rul_vals = engine_df["RUL"].values

        for i in range(len(engine_df) - window + 1):

            seq = sensor_vals[i:i+window]   # NO FLATTEN
            target = rul_vals[i+window-1]

            X.append(seq)
            y.append(target)

    return np.array(X), np.array(y)

In [4]:
window = 60

X_train, y_train = create_sequences(train_df, sensors, window)
X_test, y_test = create_sequences(test_df, sensors, window)

np.savez(
    "../data/processed/lstm_window_60.npz",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

print(X_train.shape)

(11841, 60, 10)
